# Opus-distillation training -- qwen3.5-4b (Colab)

Trains the replacement cheap-tier extraction model from the Claude-Opus-teacher
dataset built by `scripts/build_extraction_dataset.py --from-capture`. Spec:
`docs/superpowers/specs/2026-10-17-opus-distillation-design.md` section 7-8. Runbook:
`docs/superpowers/distill-runbook.md` (this notebook is one step of it).

**This repo never rents a GPU and never runs this notebook.** You run it yourself on
Colab; everything below is self-contained -- it installs its own packages, takes the
HF token from Colab secrets or an env var, and never hardcodes one.

**Pass criteria (spec section 8, checked AFTER this notebook, outside Colab):**
validity >= 99% on held-out prompts (`scripts/eval_finetune.py`), and at least as good
as solar-pro4 on every `scripts/extraction_proof.py` metric.

**Runtime:** A100 (40GB) recommended for the fastest run at `max_seq_length=20480`;
an L4 (22.5GB) works with the smaller batch size noted below. Both configs use 4-bit
loading + gradient checkpointing -- see the "what fits" note in the training cell.

**Resumable (v2, 2026-10-20).** Checkpoints, the final adapter and the eval metrics are
written to Google Drive (`DRIVE_RUN_DIR`, section 3). If the runtime dies, reconnect and
**Runtime -> Run all**: training resumes from the last checkpoint on Drive, a run whose
final adapter is already on Drive skips training altogether, and the export always
reloads that adapter from Drive -- so a death during export costs only the export.
Don't change the dataset or the training config between a crash and its resume.

## 0. Design decisions carried over from the dataset builder

- **Loss weighting -> upsampling, not a custom weighted trainer -- in EXAMPLE share
  (v2).** `build_extraction_dataset.py --upsample --upsample-basis example` repeats
  TRAIN records so `TARGET_MIX` (the per-prompt-type share: `extract_edges.edge` 34%,
  `dedupe_edges.resolve_edge` 25%, `extract_nodes.extract_summaries_batch` 18%,
  `extract_nodes.extract_text` 13%, `dedupe_nodes.nodes` 10%) is approached in EXAMPLE
  count. Why example count: Unsloth cannot count `num_items_in_batch` for
  `Qwen3_5ForConditionalGeneration` (it says so at train time), so each micro-batch loss
  is a mean over its own tokens; at `per_device_train_batch_size=1` that makes every
  EXAMPLE one equal unit of gradient, however long its answer. The v1 dataset upsampled
  in completion-TOKEN share instead, which under this trainer gave `extract_edges.edge`
  2.6% of the gradient against its 34% target and `resolve_edge` (11 copies) 82%.
  Keep the batch size at 1 -- a larger per-device batch changes the weighting again. TRL's `SFTTrainer` (which Unsloth wraps)
  has no first-class notion of a per-example loss weight it would actually honour, so
  this notebook does NOT implement a custom weighted-loss `Trainer` subclass -- it
  trains plain SFT on the **upsampled** train split and relies on repetition to do the
  weighting. `meta.loss_weight` is still read below (every record carries it, and
  `meta.repeat_index` marks each repeated copy) purely to REPORT the realised mix as a
  sanity check, not as a literal per-example multiplier.
- **Completion-only loss masking -> build with `--loss-basis completion`.**
  This notebook trains with the prompt tokens masked out of the loss (see section 9).
  `assign_loss_weights`'s docstring is explicit that the weight VALUES depend on which
  masking regime they were computed for: `basis="full"` (the builder's default) assumes
  NO completion masking, `basis="completion"` assumes completion-only masking like this
  notebook's. Since weighting here is informational only (upsampling does the real
  work, and upsampling's own `basis` argument has the same full/completion choice),
  this matters less than it would for a weighted trainer -- but the runbook still has
  you pass `--loss-basis completion` when you build, so `meta.loss_basis` in the file
  matches what this notebook actually does, and the sanity-check cell below WARNS if it
  does not.
- **Train on the assistant completion only, aligned with the THINKING-OFF SERVING
  prompt (fix round 1, 2026-10-17).** Every record's `messages` list is
  `[{role: system}, {role: user}, {role: assistant}]`. qwen3.5's chat template, called
  with `enable_thinking=False`, renders the GENERATION PROMPT for an assistant turn --
  the exact text vLLM sends the model to complete at serving time -- as
  `"<|im_start|>assistant\n<think>\n\n</think>\n\n"` (checked against
  `carev01/qwen35-4b-graphrag`'s own `chat_template.jinja`, 2026-10-17): the EMPTY think
  block is already part of the serving PROMPT, not something the model emits, so the
  loss must cover only the JSON answer and the end-of-turn marker that follow it -- not
  the think block. The SAME template can render a HISTORICAL (already-completed)
  assistant turn differently from the live generation prompt for that turn (with or
  without the empty think block), so section 7 never locates the mask boundary by
  string-searching a plain whole-conversation render; it always builds training text as
  `apply_chat_template(messages[:-1], add_generation_prompt=True, enable_thinking=
  False) + answer + "<|im_end|>\n"` -- i.e. from the SERVING prompt directly -- and
  section 8 verifies this equality against a live render before training starts,
  failing loudly if it does not hold.
- **Start from the BASE model, not the old fine-tune.** `carev01/qwen35-4b-graphrag`
  (the current production student) was distilled from solar-pro4 captures made under
  the OLD, leak-prone extraction instructions and learned that leakage (spec section 1).
  This run starts LoRA from the base `Qwen/Qwen3.5-4B` checkpoint so the new adapter is
  not conditioned on the old model's bad habit.
- **LoRA/training hyperparameters are checked, not recovered (2026-10-17).** The
  controller inspected the private HF repo `carev01/qwen35-4b-graphrag` directly --
  its README and the pushed files carry no adapter config, no trainer config, no
  recorded rank/lr/epoch/max_seq for the original solar-pro4-distilled run. Ruling:
  there is nothing to recover. Section 3's LoRA/training numbers are conservative
  Unsloth defaults for a 4B model, a stated STARTING point to tune against the held-out
  eval (section 7 of the runbook), not recovered truth.

## 1. Install packages

Current Unsloth install: `pip install unsloth` pulls its own compatible `trl`, `peft`,
`accelerate`, `bitsandbytes`, `transformers` and `datasets`; `trl`, `datasets` and
`huggingface_hub` are then requested UNPINNED (a no-op when Unsloth's pins already
satisfy them). No `%%capture` and no `--no-deps`: every pip command's output is
printed, and a failed install raises here instead of surfacing later as a confusing
import error. The installed versions are printed at the end -- section 9 feature-detects
the TRL API (`processing_class=` vs `tokenizer=`, `max_seq_length` vs `max_length`)
rather than assuming one.

In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def pip_install(*args: str) -> None:
    """Run pip, print ALL of its output, and raise on failure (never hidden)."""
    cmd = [sys.executable, "-m", "pip", "install", *args]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout, flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"pip install failed (exit {proc.returncode}): {' '.join(args)}")


if importlib.util.find_spec("unsloth") is None:
    pip_install("--upgrade", "pip")
    pip_install("unsloth")
    pip_install("trl", "datasets", "huggingface_hub")

for pkg in ("unsloth", "trl", "transformers", "peft", "datasets", "huggingface_hub"):
    print(f"{pkg}=={importlib.metadata.version(pkg)}")
print("packages ready")

## 2. HF token -- Colab secret or environment, never hardcoded

Set a Colab secret named `HF_TOKEN` (the key icon in the left sidebar) *or* export
`HF_TOKEN` in the environment before starting this runtime. This cell never prints the
token -- only whether one was found.

In [ ]:
import os


def get_hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore
        try:
            tok = userdata.get("HF_TOKEN")
            if tok:
                return tok
        except Exception:
            pass  # secret not set -- fall through to the environment
    except ImportError:
        pass  # not running on Colab
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if not tok:
        raise RuntimeError(
            "no HF token found -- add a Colab secret named HF_TOKEN, or set the "
            "HF_TOKEN environment variable, before running this cell")
    return tok


HF_TOKEN = get_hf_token()
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("HF token loaded and logged in (value never printed)")

## 3. Configuration

Edit the values in this cell for your run; nothing else below should need changing for
a normal pilot or full run.

In [ ]:
from pathlib import Path

# --- dataset source: "drive" or "hf_dataset" ---------------------------------------
DATA_SOURCE = "hf_dataset"            # or "drive"
HF_DATASET_REPO = "carev01/ft-opus-distill"   # only used when DATA_SOURCE == "hf_dataset"
HF_DATASET_REVISION = "v2"            # branch: v2 = example-share upsampling; main = v1
DRIVE_DATA_DIR = "/content/drive/MyDrive/ft-opus-v2"  # only used when DATA_SOURCE == "drive"

# --- Google Drive: checkpoints, final adapter, metrics (survive a runtime death) ----------
RUN_NAME = "opus-distill-v2"
DRIVE_RUN_DIR = Path("/content/drive/MyDrive/graph-rag-distill") / RUN_NAME
CKPT_DIR = DRIVE_RUN_DIR / "checkpoints"         # HF Trainer checkpoint-<step> folders
FINAL_ADAPTER_DIR = DRIVE_RUN_DIR / "final-adapter"  # written once training completes
FINAL_MARKER = FINAL_ADAPTER_DIR / "COMPLETE"        # written LAST: adapter fully saved
SAVE_STEPS = 25          # a checkpoint every 25 optimizer steps: ~30 min at the v1 run's
                         # pace (v2 = 364 steps over 2 epochs, ~67 s each, ~7 h in all)
SAVE_TOTAL_LIMIT = 3     # older checkpoints are deleted -> Drive trash (empty it now and then)

# --- model ----------------------------------------------------------------------------
BASE_MODEL_ID = "Qwen/Qwen3.5-4B"     # the BASE model -- see section 0's rationale
MAX_SEQ_LENGTH = 20480                # spec section 7: context of at least 20k tokens
LOAD_IN_4BIT = True                   # fits an L4 (22.5GB); set False only on a bigger GPU

# --- LoRA -- conservative Unsloth defaults for a 4B model. CHECKED, not recovered
# (2026-10-17): the controller inspected carev01/qwen35-4b-graphrag directly (README +
# pushed files) for the original solar-pro4-distilled run's rank/lr/epoch/max_seq and
# found none recorded -- there is nothing to recover. Treat these as a starting point to
# tune against the held-out eval (runbook section 7), not as recovered truth.
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                       "gate_proj", "up_proj", "down_proj"]

# --- training ---------------------------------------------------------------------------
LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 2
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01
LR_SCHEDULER_TYPE = "cosine"
SEED = 20261017

# per-device batch size x grad accumulation: what fits at MAX_SEQ_LENGTH=20480 with
# gradient checkpointing on. These are STARTING points -- Unsloth's OOM message tells
# you which way to move them:
#   A100 40GB, 4-bit, grad checkpointing: per_device=1, grad_accum=16 (effective batch 16)
#   L4 22.5GB,  4-bit, grad checkpointing: per_device=1, grad_accum=32 (effective batch 32)
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16       # raise to 32 on an L4
PER_DEVICE_EVAL_BATCH_SIZE = 1

# --- export -----------------------------------------------------------------------------
HF_MODEL_REPO = "carev01/qwen35-4b-graphrag"
HF_MODEL_REVISION = "opus-distill-2026-10"
HF_MODEL_PRIVATE = False   # the repo is public by design (since 2026-10-09: Vast pulls it
                           # without a token); a branch cannot be more private than its repo

OUT_DIR = Path("/content/outputs/opus-distill")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("config set")

## 3a. Mount Google Drive

Checkpoints, the final adapter and the eval metrics go to `DRIVE_RUN_DIR`, never to the
runtime's local disk, so a dead runtime loses at most `SAVE_STEPS` steps. Lists what is
already there, so you can see at a glance whether this run will start fresh, resume, or
skip straight to export.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
_ckpts = sorted(CKPT_DIR.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
print(f"run dir: {DRIVE_RUN_DIR}")
print(f"checkpoints on Drive: {[p.name for p in _ckpts] or 'none'}")
print(f"final adapter on Drive (complete): {FINAL_MARKER.exists()}")

## 4. Get the dataset

Either mount Drive and point at a folder already holding `train.jsonl` / `val.jsonl` /
`MANIFEST.json` (the output of `scripts/distill/package_dataset.py`, unzipped), or
download the same three files from an HF **dataset** repo -- whichever the runbook's
packaging step pushed to.

In [ ]:
DATA_DIR = Path("/content/data/ft-opus")
DATA_DIR.mkdir(parents=True, exist_ok=True)

if DATA_SOURCE == "drive":
    DATA_DIR = Path(DRIVE_DATA_DIR)  # Drive is mounted in section 3a
elif DATA_SOURCE == "hf_dataset":
    from huggingface_hub import hf_hub_download
    for fname in ("train.jsonl", "val.jsonl", "MANIFEST.json", "sample.json"):
        try:
            p = hf_hub_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                                filename=fname, revision=HF_DATASET_REVISION, token=HF_TOKEN)
        except Exception as e:
            if fname == "sample.json":
                print(f"  {fname} not in {HF_DATASET_REPO} (optional, provenance only) -- skipping")
                continue
            raise
        (DATA_DIR / fname).write_bytes(Path(p).read_bytes())
else:
    raise ValueError(f"unknown DATA_SOURCE {DATA_SOURCE!r}")

assert (DATA_DIR / "train.jsonl").exists(), f"no train.jsonl under {DATA_DIR}"
assert (DATA_DIR / "val.jsonl").exists(), f"no val.jsonl under {DATA_DIR}"
print(f"dataset ready at {DATA_DIR}")

## 5. Load train/val and report the realised mix

Reads `meta.loss_weight` / `meta.loss_basis` / `meta.repeat_index` exactly as
`build_extraction_dataset.py` writes them (see its `_record` and `assign_loss_weights`
docstrings) and prints the per-prompt-type share -- purely a sanity check (section 0) -- the `share` column is the example share, which is
this trainer's gradient share:
training applies no per-example weight directly, so this cell is how you SEE the mix
the upsampling built, not how you apply it.

In [ ]:
import json
from collections import Counter, defaultdict


def read_jsonl(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_records = read_jsonl(DATA_DIR / "train.jsonl")
val_records = read_jsonl(DATA_DIR / "val.jsonl")
print(f"{len(train_records)} train records, {len(val_records)} val records")

bases = {r["meta"].get("loss_basis") for r in train_records}
if bases - {"completion"}:
    print(f"WARNING: train.jsonl was built with loss_basis={bases!r}, not "
          "'completion' -- this notebook masks prompt tokens (completion-only loss), "
          "so meta.loss_weight's VALUES assumed the wrong masking regime when the "
          "dataset was built. Rebuild with --loss-basis completion if you intend to "
          "act on loss_weight directly; harmless as-is since this notebook relies on "
          "upsampling, not loss_weight, for the mix.")

ubases = {r["meta"].get("upsample_basis") for r in train_records}
if ubases != {"example"}:
    print(f"WARNING: train.jsonl was upsampled with basis={ubases!r}, not 'example'. "
          "This trainer weights every example equally (batch size 1, per-example mean "
          "loss -- section 0), so only an example-share upsample hits TARGET_MIX. "
          "Rebuild with --upsample --upsample-basis example.")

counts = Counter(r["prompt_name"] for r in train_records)
weights = {r["prompt_name"]: r["meta"].get("loss_weight") for r in train_records}
repeats = defaultdict(int)
for r in train_records:
    repeats[r["prompt_name"]] = max(repeats[r["prompt_name"]],
                                     r["meta"].get("repeat_index", 0) + 1)
total = sum(counts.values())
# 'share' is the example share -- with this trainer, the gradient share (section 0).
print(f"{'prompt_name':45s} {'n':>6} {'share':>7} {'loss_weight':>12} {'max_repeat':>11}")
for name, n in sorted(counts.items(), key=lambda kv: -kv[1]):
    print(f"{name:45s} {n:6d} {100*n/total:6.1f}% {weights.get(name):12} {repeats[name]:11d}")

## 6. Load the base model and apply LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,  # auto: bf16 where supported, else fp16
    token=HF_TOKEN,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",  # long-context checkpointing, needed at 20k tokens
    random_state=SEED,
)
print("base model loaded, LoRA adapter attached")

## 7. Build the chat-formatted dataset, aligned with the thinking-off serving prompt

Each record's `messages` is `[system, user, assistant]` (section 0). This notebook does
NOT render the plain whole-conversation template and hope the assistant turn starts
where expected -- it builds the training text the same way section 0 describes:
`serving_prompt(messages[:-1]) + answer + "<|im_end|>\n"`, where `serving_prompt` is
`apply_chat_template(..., add_generation_prompt=True, enable_thinking=False)`, i.e. the
literal text vLLM sends the model to complete in production. This mirrors
`scripts/distill/chat_format.py` in the repo (duplicated inline here so this notebook
needs no clone of the private repo on Colab) -- see that module's docstring for why a
plain render is not trustworthy for locating the mask boundary. `path` records, per
example, whether this tokenizer's plain render happened to agree (`"plain"`) or had to
be overridden (`"generation_prompt"`) -- bookkeeping only, `text` is always the
constructed form either way.

In [ ]:
from datasets import Dataset

END_OF_TURN = "<|im_end|>\n"


def build_training_text(messages: list[dict]) -> tuple[str, int, str]:
    """Mirrors scripts/distill/chat_format.py:build_training_text. Returns
    (text, mask_start, path) -- see section 0 and that module's docstring."""
    assert messages[-1]["role"] == "assistant", messages
    answer = messages[-1]["content"]
    prefix = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True,
        enable_thinking=False)
    constructed = prefix + answer + END_OF_TURN
    plain = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False, enable_thinking=False)
    path = "plain" if plain == constructed else "generation_prompt"
    return constructed, len(prefix), path


def to_text(example: dict) -> dict:
    text, mask_start, path = build_training_text(example["messages"])
    return {"text": text, "mask_start": mask_start, "path": path}


train_ds = Dataset.from_list(train_records).map(to_text, remove_columns=["messages"])
val_ds = Dataset.from_list(val_records).map(to_text, remove_columns=["messages"])

path_counts = Counter(train_ds["path"])
print(f"construction path across train: {dict(path_counts)}")
print(train_ds[0]["text"][:800])

train_ds = train_ds.remove_columns(["mask_start", "path"])
val_ds = val_ds.remove_columns(["mask_start", "path"])

## 8. Verify: training text == serving prompt + answer + end-of-turn (fail loudly)

Renders ONE training example's prefix as the live SERVING prompt
(`add_generation_prompt=True, enable_thinking=False`) and checks, independently of
section 7's helper, that `build_training_text`'s output is exactly that serving prompt
plus the answer plus the end-of-turn marker -- i.e. the model would be trained on
precisely what it is asked to complete when served, nothing more and nothing less. This
is the one cell in the notebook that must not be skipped: an assertion failure here
means this tokenizer's real chat template does not behave the way section 0 assumes,
and training would silently teach the wrong completion boundary.

In [ ]:
_probe = train_records[0]
_serving = tokenizer.apply_chat_template(
    _probe["messages"][:-1], tokenize=False, add_generation_prompt=True,
    enable_thinking=False)
_answer = _probe["messages"][-1]["content"]
_expected = _serving + _answer + END_OF_TURN
_built_text, _built_mask_start, _built_path = build_training_text(_probe["messages"])

if _built_text != _expected:
    _divergence = next(
        (i for i in range(min(len(_built_text), len(_expected)))
         if _built_text[i] != _expected[i]), None)
    raise AssertionError(
        "training text does not equal serving prompt + answer + end-of-turn marker -- "
        "training and serving would NOT target the same completion. "
        f"path={_built_path!r}, first divergence at char {_divergence!r}. "
        f"built[:200]={_built_text[:200]!r} expected[:200]={_expected[:200]!r}")
assert _built_mask_start == len(_serving), (
    f"mask_start {_built_mask_start} != len(serving prompt) {len(_serving)}")

print(f"OK: training text == serving prompt + answer + end-of-turn (path={_built_path})")
print("serving prompt tail:", repr(_serving[-70:]))

## 9. Train -- completion-only loss, eval on val

`train_on_responses_only` masks every token outside the assistant turn. `RESPONSE_PART`
is derived from the tokenizer's OWN template (a one-token probe rendered with
`add_generation_prompt=True, enable_thinking=False`), not hardcoded, so it is exactly
the serving-prompt suffix section 8 just verified -- including the empty think block --
rather than a guess at what the template looks like.

The trainer is built against whichever TRL the install cell produced: `processing_class=`
when `SFTTrainer` takes it (current TRL), else `tokenizer=`; `max_seq_length` or its
successor `max_length` on `SFTConfig`, whichever exists. Both are feature-detected.

**Mask check (fails loudly).** After `train_on_responses_only`, the unmasked labels of
one training example are decoded and must equal that example's answer plus
`<|im_end|>` (a trailing newline allowed, per the template's `"<|im_end|>\n"`). Anything
else means the loss would cover the wrong tokens; both strings are printed and the cell
raises before any GPU time is spent on training.

**Checkpoints** go to `CKPT_DIR` on Drive every `SAVE_STEPS` optimizer steps: the LoRA
weights plus the optimizer, scheduler, RNG and trainer state, which is everything
`trainer.train(resume_from_checkpoint=...)` needs to continue exactly where it stopped.

In [ ]:
import dataclasses
import inspect

from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

_probe_prefix = tokenizer.apply_chat_template(
    [{"role": "user", "content": "x"}], tokenize=False, add_generation_prompt=True,
    enable_thinking=False)
RESPONSE_PART = _probe_prefix.rsplit(END_OF_TURN, 1)[-1]
INSTRUCTION_PART = "<|im_start|>user\n"
print(f"instruction_part={INSTRUCTION_PART!r} response_part={RESPONSE_PART!r}")

# TRL renamed SFTConfig.max_seq_length to max_length; use whichever this version has.
_config_fields = {f.name for f in dataclasses.fields(SFTConfig)}
_max_len_field = "max_seq_length" if "max_seq_length" in _config_fields else "max_length"
print(f"SFTConfig sequence-length field: {_max_len_field}")

sft_config = SFTConfig(
    output_dir=str(CKPT_DIR),  # on Drive (section 3a) -- what makes the run resumable
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    optim="adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    dataset_text_field="text",
    packing=False,  # required for train_on_responses_only's per-example masking
    seed=SEED,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    report_to="none",
    **{_max_len_field: MAX_SEQ_LENGTH},
)

# Current TRL takes the tokenizer as processing_class=; older TRL as tokenizer=.
_trainer_params = inspect.signature(SFTTrainer.__init__).parameters
_tokenizer_kwarg = "processing_class" if "processing_class" in _trainer_params else "tokenizer"
print(f"SFTTrainer tokenizer argument: {_tokenizer_kwarg}=")

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=sft_config,
    **{_tokenizer_kwarg: tokenizer},
)

trainer = train_on_responses_only(
    trainer,
    instruction_part=INSTRUCTION_PART,
    response_part=RESPONSE_PART,
)

# --- mask check: the loss covers exactly answer + "<|im_end|>" ---------------------
_labels = list(trainer.train_dataset[0]["labels"])
_unmasked = [t for t in _labels if t != -100]
_decoded = tokenizer.decode(_unmasked, skip_special_tokens=False)
_expected = train_records[0]["messages"][-1]["content"] + "<|im_end|>"
if _decoded not in (_expected, _expected + "\n"):
    print("decoded unmasked labels:")
    print(repr(_decoded))
    print("expected (answer + '<|im_end|>', optional trailing newline):")
    print(repr(_expected))
    raise AssertionError(
        "train_on_responses_only did not leave exactly answer + '<|im_end|>' unmasked "
        "-- the loss would cover the wrong tokens. Both strings are printed above; "
        "check INSTRUCTION_PART / RESPONSE_PART against this tokenizer's template.")
print(f"OK: unmasked labels == answer + '<|im_end|>' "
      f"({len(_unmasked)} of {len(_labels)} tokens carry loss)")

In [ ]:
import json as _json

_CKPT_FILES = ("trainer_state.json", "optimizer.pt", "scheduler.pt")


def last_complete_checkpoint(ckpt_dir: Path) -> str | None:
    """Newest checkpoint-<step> holding everything a resume needs. A runtime that died
    mid-save leaves a partial folder that get_last_checkpoint would still pick (and the
    resume would then fail); skip it and fall back to the previous one."""
    for c in sorted(ckpt_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]),
                    reverse=True):
        if all((c / f).exists() for f in _CKPT_FILES) and any(c.glob("adapter_model.*")):
            return str(c)
        print(f"skipping incomplete {c.name}")
    return None


if FINAL_MARKER.exists():
    print(f"final adapter already on Drive ({FINAL_ADAPTER_DIR}) -- training is done, "
          "skipping to export. Delete that folder to train again.")
    TRAINED_THIS_SESSION = False
else:
    _last = last_complete_checkpoint(CKPT_DIR)
    print(f"resuming from {_last}" if _last else "no checkpoint on Drive -- starting fresh")
    train_result = trainer.train(resume_from_checkpoint=_last)
    print(train_result)
    model.save_pretrained(str(FINAL_ADAPTER_DIR))
    tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
    (DRIVE_RUN_DIR / "train_result.json").write_text(
        _json.dumps(train_result.metrics, indent=2))
    FINAL_MARKER.write_text("adapter + tokenizer fully saved\n")
    print(f"final adapter saved -> {FINAL_ADAPTER_DIR}")
    TRAINED_THIS_SESSION = True

In [ ]:
if TRAINED_THIS_SESSION:
    eval_metrics = trainer.evaluate()
    (DRIVE_RUN_DIR / "eval_metrics.json").write_text(_json.dumps(eval_metrics, indent=2))
    print(eval_metrics)
else:
    _saved = DRIVE_RUN_DIR / "eval_metrics.json"
    print(_saved.read_text() if _saved.exists() else "no eval metrics saved for this run")

## 10. Export -- merged safetensors + Q8 GGUF, new HF revision

Always starts from the final adapter ON DRIVE (reloaded onto the base model), whether
training finished a minute ago or in a runtime that has since died -- one export path,
the same every time. A three-record smoke test checks the reloaded model still answers in
JSON before anything is uploaded.

Pushes to `carev01/qwen35-4b-graphrag`, revision `opus-distill-2026-10` -- a NEW
revision, so the current production revision stays untouched and available for
comparison (spec section 7: "the old model stays available for comparison"). Files are
saved locally once and uploaded with `HfApi` to that revision explicitly -- no second
merge inside a `push_to_hub_*` helper.

In [ ]:
import gc

from unsloth import FastLanguageModel

for _name in ("trainer", "model"):
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()

assert FINAL_MARKER.exists(), (
    f"no complete final adapter at {FINAL_ADAPTER_DIR} -- run section 9 first")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(FINAL_ADAPTER_DIR),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=None,
    token=HF_TOKEN,
)
print(f"reloaded base + adapter from {FINAL_ADAPTER_DIR}")

# --- smoke test: three short held-out prompts must come back as JSON objects ------------
FastLanguageModel.for_inference(model)
_short = sorted(val_records, key=lambda r: len(r["messages"][1]["content"]))[:3]
for _r in _short:
    _prompt = tokenizer.apply_chat_template(
        _r["messages"][:-1], tokenize=False, add_generation_prompt=True,
        enable_thinking=False)
    _ids = tokenizer(text=_prompt, return_tensors="pt").to(model.device)
    _out = model.generate(**_ids, max_new_tokens=512, do_sample=False)
    _reply = tokenizer.decode(_out[0][_ids["input_ids"].shape[1]:], skip_special_tokens=True)
    try:
        _ok = isinstance(_json.loads(_reply), dict)
    except ValueError:
        _ok = False
    print(f"{_r['prompt_name']:40s} json={_ok} reply={_reply[:120]!r}")
    assert _ok, "the reloaded model did not answer in JSON -- do not upload this export"
print("smoke test passed")

In [ ]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE,
                exist_ok=True)
api.create_branch(repo_id=HF_MODEL_REPO, repo_type="model", branch=HF_MODEL_REVISION,
                  exist_ok=True)

merged_dir = OUT_DIR / "merged-safetensors"
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method="merged_16bit")
_index = json.loads((merged_dir / "model.safetensors.index.json").read_text())
_missing = sorted(set(_index["weight_map"].values())
                  - {p.name for p in merged_dir.glob("*.safetensors")})
assert not _missing, f"merged save is missing shards: {_missing}"

api.upload_folder(repo_id=HF_MODEL_REPO, repo_type="model", folder_path=str(merged_dir),
                  revision=HF_MODEL_REVISION,
                  commit_message=f"{RUN_NAME}: merged bf16 safetensors")
print(f"merged safetensors pushed -> {HF_MODEL_REPO}@{HF_MODEL_REVISION}")

In [ ]:
gguf_dir = OUT_DIR / "gguf-q8"
model.save_pretrained_gguf(str(gguf_dir), tokenizer, quantization_method="q8_0")

# Unsloth versions differ on where the .gguf lands (the folder itself or a "_gguf"
# sibling) and may leave an intermediate BF16 file next to it: upload only the Q8_0.
_ggufs = sorted({p for p in OUT_DIR.glob("gguf-q8*/**/*.gguf")})
_q8 = [p for p in _ggufs if "q8_0" in p.name.lower()] or (_ggufs if len(_ggufs) == 1 else [])
assert _q8, f"no Q8_0 .gguf found under {OUT_DIR} (found: {[p.name for p in _ggufs]})"
for _f in _q8:
    api.upload_file(path_or_fileobj=str(_f), path_in_repo=_f.name, repo_id=HF_MODEL_REPO,
                    repo_type="model", revision=HF_MODEL_REVISION,
                    commit_message=f"{RUN_NAME}: Q8_0 GGUF")
    print(f"{_f.name} ({_f.stat().st_size / 1e9:.1f} GB) pushed -> "
          f"{HF_MODEL_REPO}@{HF_MODEL_REVISION}")

## 11. Next steps -- outside this notebook

This notebook's job ends here. Hand the revision name (`opus-distill-2026-10`) back to
the runbook for evaluation:

1. `scripts/eval_finetune.py --val <DATA_DIR>/val.jsonl --model <served-name>
   --base-url <server>/v1/chat/completions --per-prompt 0` against a server serving
   this revision -- validity rate over the FULL held-out set (`--per-prompt 0`; the
   default 25 per prompt type is a subsample), compared with the teacher.
2. `scripts/extraction_proof.py ... --out report.md` with `CHEAP_LLM_BASE_URL` /
   `CHEAP_LLM_MODEL` pointed at that same server, compared against solar-pro4 and the
   old fine-tune (production's current revision). A PAID run: ask the user first.

See `docs/superpowers/distill-runbook.md` section "Evaluate" for the exact commands and
the pass criteria (spec section 8).